# Day 1: Markov Decision Processes & Dynamic Programming

**Reinforcement Learning: From Theory to Practice**

---

## Overview

In this notebook we build the mathematical foundations of reinforcement learning:

1. **Markov Decision Processes (MDPs)** -- the formal framework for sequential decision-making
2. **Policy Evaluation** -- computing the value function for a given policy
3. **Policy Iteration** -- alternating evaluation and improvement to find optimal policies
4. **Value Iteration** -- a more direct route to the optimal value function

We implement everything from scratch on a GridWorld environment and visualize the results.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib import cm

np.set_printoptions(precision=3, suppress=True)
%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 12

## 1. GridWorld Environment

We define a simple $N \times N$ grid. The agent can move in four directions: **up, down, left, right**.

- Stepping off the grid leaves the agent in place and yields reward $-1$.
- Reaching the **goal cell** yields reward $+1$ and terminates the episode.
- All other transitions yield reward $-0.04$ (a small penalty to encourage efficiency).
- Optional **walls** block movement.

The MDP is defined by $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$.

In [ ]:
class GridWorld:
    """
    A deterministic GridWorld MDP.
    
    States are indexed as integers 0 .. (rows*cols - 1).
    Actions: 0=up, 1=right, 2=down, 3=left
    """
    
    ACTIONS = {0: (-1, 0), 1: (0, 1), 2: (1, 0), 3: (0, -1)}
    ACTION_NAMES = {0: 'Up', 1: 'Right', 2: 'Down', 3: 'Left'}
    ACTION_SYMBOLS = {0: '\u2191', 1: '\u2192', 2: '\u2193', 3: '\u2190'}  # arrows
    
    def __init__(self, rows=4, cols=4, goal=(0, 3), walls=None,
                 step_reward=-0.04, goal_reward=1.0, wall_reward=-1.0,
                 gamma=0.99, noise=0.0):
        self.rows = rows
        self.cols = cols
        self.goal = goal
        self.walls = set(walls) if walls else set()
        self.step_reward = step_reward
        self.goal_reward = goal_reward
        self.wall_reward = wall_reward
        self.gamma = gamma
        self.noise = noise  # probability of slipping to a random action
        
        self.n_states = rows * cols
        self.n_actions = 4
        
        # Pre-compute the non-wall states
        self.states = [s for s in range(self.n_states)
                       if self._to_rc(s) not in self.walls]
        self.terminal_states = {self._to_idx(*goal)}
    
    # ---- coordinate helpers ----
    def _to_rc(self, s):
        return (s // self.cols, s % self.cols)
    
    def _to_idx(self, r, c):
        return r * self.cols + c
    
    def _is_valid(self, r, c):
        return (0 <= r < self.rows and 0 <= c < self.cols
                and (r, c) not in self.walls)
    
    # ---- MDP dynamics ----
    def get_transitions(self, s, a):
        """
        Returns list of (probability, next_state, reward, done) tuples.
        """
        if s in self.terminal_states:
            return [(1.0, s, 0.0, True)]
        
        r, c = self._to_rc(s)
        if (r, c) in self.walls:
            return [(1.0, s, 0.0, True)]
        
        transitions = []
        
        # Intended action + noise
        if self.noise > 0:
            action_probs = []
            for a_prime in range(self.n_actions):
                if a_prime == a:
                    action_probs.append((1.0 - self.noise, a_prime))
                else:
                    action_probs.append((self.noise / 3.0, a_prime))
        else:
            action_probs = [(1.0, a)]
        
        for prob, act in action_probs:
            dr, dc = self.ACTIONS[act]
            nr, nc = r + dr, c + dc
            if self._is_valid(nr, nc):
                ns = self._to_idx(nr, nc)
                if (nr, nc) == self.goal:
                    transitions.append((prob, ns, self.goal_reward, True))
                else:
                    transitions.append((prob, ns, self.step_reward, False))
            else:
                # Bump into wall or boundary -- stay in place
                transitions.append((prob, s, self.wall_reward if not self._is_valid(nr, nc) and (nr, nc) in self.walls else self.step_reward, False))
        
        # Merge transitions to same next_state
        merged = {}
        for prob, ns, rew, done in transitions:
            if ns in merged:
                old_p, old_r, old_d = merged[ns]
                merged[ns] = (old_p + prob, 
                              (old_r * old_p + rew * prob) / (old_p + prob),
                              done)
            else:
                merged[ns] = (prob, rew, done)
        
        return [(p, ns, r, d) for ns, (p, r, d) in merged.items()]
    
    def render_values(self, V, title='State Values'):
        """Render value function as a heatmap."""
        grid = np.full((self.rows, self.cols), np.nan)
        for s in self.states:
            r, c = self._to_rc(s)
            grid[r, c] = V[s]
        
        fig, ax = plt.subplots(1, 1, figsize=(self.cols + 2, self.rows + 1))
        masked = np.ma.masked_invalid(grid)
        cmap = cm.RdYlGn
        cmap.set_bad('gray', 0.3)
        im = ax.imshow(masked, cmap=cmap, interpolation='nearest')
        
        for r in range(self.rows):
            for c in range(self.cols):
                if not np.isnan(grid[r, c]):
                    ax.text(c, r, f'{grid[r, c]:.2f}', ha='center', va='center',
                            fontsize=11, fontweight='bold')
                elif (r, c) in self.walls:
                    ax.text(c, r, 'WALL', ha='center', va='center',
                            fontsize=9, color='white')
        
        # Mark goal
        gr, gc = self.goal
        ax.add_patch(plt.Rectangle((gc - 0.5, gr - 0.5), 1, 1,
                                    fill=False, edgecolor='gold', linewidth=3))
        
        ax.set_xticks(range(self.cols))
        ax.set_yticks(range(self.rows))
        ax.set_title(title, fontsize=14)
        plt.colorbar(im, ax=ax, shrink=0.8)
        plt.tight_layout()
        plt.show()
    
    def render_policy(self, policy, title='Policy'):
        """Render a deterministic policy as arrows on a grid."""
        fig, ax = plt.subplots(1, 1, figsize=(self.cols + 1, self.rows + 1))
        
        # Draw grid
        for r in range(self.rows):
            for c in range(self.cols):
                if (r, c) in self.walls:
                    ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1,
                                                color='gray', alpha=0.5))
                elif (r, c) == self.goal:
                    ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1,
                                                color='gold', alpha=0.4))
                    ax.text(c, r, 'G', ha='center', va='center',
                            fontsize=16, fontweight='bold', color='darkgreen')
                else:
                    s = self._to_idx(r, c)
                    ax.text(c, r, self.ACTION_SYMBOLS[policy[s]],
                            ha='center', va='center', fontsize=20)
        
        ax.set_xlim(-0.5, self.cols - 0.5)
        ax.set_ylim(self.rows - 0.5, -0.5)
        ax.set_xticks(range(self.cols))
        ax.set_yticks(range(self.rows))
        ax.grid(True, linewidth=2)
        ax.set_title(title, fontsize=14)
        ax.set_aspect('equal')
        plt.tight_layout()
        plt.show()


# Instantiate a 4x4 GridWorld
env = GridWorld(rows=4, cols=4, goal=(0, 3), walls=[(1, 1)],
                step_reward=-0.04, goal_reward=1.0, gamma=0.99)

print(f'Grid: {env.rows}x{env.cols}')
print(f'Number of states: {env.n_states}')
print(f'Goal: {env.goal}')
print(f'Walls: {env.walls}')
print(f'Discount factor: {env.gamma}')

## 2. Policy Evaluation (Iterative)

Given a fixed policy $\pi$, we compute its value function $V^\pi$ via the **Bellman expectation equation**:

$$V^\pi(s) = \sum_a \pi(a|s) \sum_{s', r} p(s', r | s, a) \left[ r + \gamma V^\pi(s') \right]$$

We iterate until convergence (the maximum change across all states falls below a threshold $\theta$).

In [ ]:
def policy_evaluation(env, policy, theta=1e-6, max_iters=1000):
    """
    Iterative policy evaluation for a deterministic policy.
    
    Parameters
    ----------
    env : GridWorld
    policy : array of shape (n_states,) with action indices
    theta : convergence threshold
    max_iters : safety limit on iterations
    
    Returns
    -------
    V : array of shape (n_states,)
    history : list of max deltas per iteration
    """
    V = np.zeros(env.n_states)
    history = []
    
    for i in range(max_iters):
        delta = 0.0
        for s in env.states:
            if s in env.terminal_states:
                continue
            
            v_old = V[s]
            a = policy[s]
            
            # Bellman expectation backup
            v_new = 0.0
            for prob, ns, reward, done in env.get_transitions(s, a):
                v_new += prob * (reward + env.gamma * V[ns] * (1 - done))
            
            V[s] = v_new
            delta = max(delta, abs(v_old - v_new))
        
        history.append(delta)
        if delta < theta:
            print(f'Policy evaluation converged in {i+1} iterations (delta={delta:.2e})')
            break
    
    return V, history


# Evaluate a random policy (all actions = RIGHT)
random_policy = np.ones(env.n_states, dtype=int)  # all go right
V_rand, hist_rand = policy_evaluation(env, random_policy)

env.render_values(V_rand, title='Value Function (All-Right Policy)')

In [ ]:
# Plot convergence
plt.figure(figsize=(8, 4))
plt.semilogy(hist_rand, linewidth=2)
plt.xlabel('Iteration')
plt.ylabel('Max |delta V|')
plt.title('Policy Evaluation Convergence')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Policy Iteration

Policy iteration alternates between:
1. **Policy Evaluation** -- compute $V^\pi$ for the current policy
2. **Policy Improvement** -- make the policy greedy w.r.t. $V^\pi$

$$\pi'(s) = \arg\max_a \sum_{s',r} p(s',r|s,a)\left[r + \gamma V^\pi(s')\right]$$

This is guaranteed to converge to the optimal policy $\pi^*$.

In [ ]:
def policy_improvement(env, V):
    """
    Compute a greedy policy w.r.t. value function V.
    
    Returns
    -------
    policy : array of shape (n_states,)
    """
    policy = np.zeros(env.n_states, dtype=int)
    
    for s in env.states:
        if s in env.terminal_states:
            continue
        
        q_values = np.zeros(env.n_actions)
        for a in range(env.n_actions):
            for prob, ns, reward, done in env.get_transitions(s, a):
                q_values[a] += prob * (reward + env.gamma * V[ns] * (1 - done))
        
        policy[s] = np.argmax(q_values)
    
    return policy


def policy_iteration(env, theta=1e-6, max_iters=100):
    """
    Full policy iteration algorithm.
    
    Returns
    -------
    policy : optimal policy
    V : optimal value function
    iteration_count : number of policy iteration steps
    """
    # Start with a random policy
    policy = np.random.randint(0, env.n_actions, size=env.n_states)
    
    for i in range(max_iters):
        # Evaluate
        V, _ = policy_evaluation(env, policy, theta=theta)
        
        # Improve
        new_policy = policy_improvement(env, V)
        
        # Check for convergence
        if np.array_equal(policy, new_policy):
            print(f'\nPolicy iteration converged in {i+1} outer iterations.')
            return new_policy, V, i + 1
        
        policy = new_policy
    
    print(f'Policy iteration reached max iterations ({max_iters}).')
    return policy, V, max_iters


pi_star, V_star_pi, n_iters = policy_iteration(env)

In [ ]:
# Visualize optimal policy and value function from Policy Iteration
env.render_values(V_star_pi, title='Optimal Value Function (Policy Iteration)')
env.render_policy(pi_star, title='Optimal Policy (Policy Iteration)')

## 4. Value Iteration

Instead of full policy evaluation at each step, value iteration applies the **Bellman optimality backup** directly:

$$V_{k+1}(s) = \max_a \sum_{s',r} p(s',r|s,a)\left[r + \gamma V_k(s')\right]$$

This converges to $V^*$ and is often faster in practice.

In [ ]:
def value_iteration(env, theta=1e-6, max_iters=1000):
    """
    Value iteration algorithm.
    
    Returns
    -------
    V : optimal value function
    policy : derived optimal policy
    history : list of (iteration, max_delta) for convergence tracking
    """
    V = np.zeros(env.n_states)
    history = []
    
    for i in range(max_iters):
        delta = 0.0
        for s in env.states:
            if s in env.terminal_states:
                continue
            
            v_old = V[s]
            
            # Bellman optimality backup
            q_values = np.zeros(env.n_actions)
            for a in range(env.n_actions):
                for prob, ns, reward, done in env.get_transitions(s, a):
                    q_values[a] += prob * (reward + env.gamma * V[ns] * (1 - done))
            
            V[s] = np.max(q_values)
            delta = max(delta, abs(v_old - V[s]))
        
        history.append(delta)
        if delta < theta:
            print(f'Value iteration converged in {i+1} iterations (delta={delta:.2e})')
            break
    
    # Extract policy
    policy = policy_improvement(env, V)
    
    return V, policy, history


V_star_vi, pi_star_vi, hist_vi = value_iteration(env)

In [ ]:
env.render_values(V_star_vi, title='Optimal Value Function (Value Iteration)')
env.render_policy(pi_star_vi, title='Optimal Policy (Value Iteration)')

In [ ]:
# Compare convergence of value iteration
plt.figure(figsize=(8, 4))
plt.semilogy(hist_vi, linewidth=2, color='darkorange')
plt.xlabel('Iteration')
plt.ylabel('Max |delta V|')
plt.title('Value Iteration Convergence')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Verify both methods agree
print('Max difference between PI and VI value functions:',
      np.max(np.abs(V_star_pi - V_star_vi)))
print('Policies agree:', np.array_equal(pi_star, pi_star_vi))

## 5. Larger GridWorld with Stochastic Dynamics

Let us test on a bigger grid with **stochastic transitions** (the agent slips with probability 0.2).

In [ ]:
# 6x6 grid with walls and stochastic dynamics
env_large = GridWorld(
    rows=6, cols=6,
    goal=(0, 5),
    walls=[(1, 2), (2, 2), (3, 4), (4, 1), (4, 2)],
    step_reward=-0.04,
    goal_reward=1.0,
    gamma=0.95,
    noise=0.2
)

V_large, pi_large, hist_large = value_iteration(env_large)

env_large.render_values(V_large, title='Value Function -- 6x6 Stochastic Grid')
env_large.render_policy(pi_large, title='Optimal Policy -- 6x6 Stochastic Grid')

## 6. Effect of Discount Factor $\gamma$

The discount factor controls how much the agent values future rewards. Let us visualize the optimal value function for different values of $\gamma$.

In [ ]:
gammas = [0.5, 0.8, 0.95, 0.99]

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, gamma in zip(axes, gammas):
    env_g = GridWorld(rows=4, cols=4, goal=(0, 3), walls=[(1, 1)],
                      step_reward=-0.04, goal_reward=1.0, gamma=gamma)
    V_g, _, _ = value_iteration(env_g)
    
    grid = np.full((env_g.rows, env_g.cols), np.nan)
    for s in env_g.states:
        r, c = env_g._to_rc(s)
        grid[r, c] = V_g[s]
    
    masked = np.ma.masked_invalid(grid)
    im = ax.imshow(masked, cmap='RdYlGn', interpolation='nearest')
    for r in range(env_g.rows):
        for c in range(env_g.cols):
            if not np.isnan(grid[r, c]):
                ax.text(c, r, f'{grid[r,c]:.2f}', ha='center', va='center', fontsize=9)
    ax.set_title(f'$\\gamma = {gamma}$', fontsize=13)
    ax.set_xticks(range(env_g.cols))
    ax.set_yticks(range(env_g.rows))

plt.suptitle('Effect of Discount Factor on Value Function', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## Summary

| Method | Idea | Convergence |
|--------|------|-------------|
| **Policy Evaluation** | Compute $V^\pi$ for fixed $\pi$ | Bellman expectation equation |
| **Policy Iteration** | Alternate evaluation + greedy improvement | Finite # of policies => terminates |
| **Value Iteration** | Bellman optimality backup directly | Contraction mapping theorem |

### Key Takeaways

- Dynamic programming methods require a **complete model** of the environment (transition probabilities).
- The Bellman operator is a **contraction mapping** (we will revisit this from a topological perspective on Day 5).
- Value iteration is typically faster per iteration but may need more iterations.
- The discount factor $\gamma$ controls the planning horizon.

**Next:** Day 2 -- Model-free methods (Monte Carlo, TD learning, SARSA, Q-Learning)